# Experiment 001 — Typo Robustness: Colab Pilot

Full pilot pipeline in six cells:
**clone → install → build items → build dictionary → generate → analyze + download**

Scoring is performed inline during generation (no separate scoring step).
Uses `qwen_1b5_pilot` (Qwen2.5-1.5B-Instruct, ungated, no HF auth needed) on a Colab T4.
The pilot config (`configs/pilot.yaml`) sets `is_confirmatory: false`, so pinned model
revisions are not enforced.

**Before starting:** Runtime → Change runtime type → T4 GPU.

---
### Pipeline overview

| Cell | Tool | Time (approx) | Output |
|------|------|---------------|--------|
| 1 | clone + CPU deps | 2 min | environment ready |
| 2 | GPU stack + spaCy transformer | 5 min | CUDA verified |
| 3 | `build_task_items` + `build_annotated_dataset` | 5–10 min | `data/items/*.jsonl` |
| 4 | `build_dictionary` (SCOWL) | 1 min | `data/wordlists/en_us_pinned.txt` |
| 5 | `run_generation` | 30–60 min | `results/pilot/pilot_generations.jsonl` |
| 6 | `run_analysis` + `build_report` + download | 5 min | `pilot_results.zip` |

---
### Key outputs to review after Cell 6

- **Discordant rate per cell** → sets N for the main run (Connor 1987; design/06 §6.3)
- **Clean accuracy A₀** → validate against expected GSM-Symbolic / MMLU bands
- **`max_new_tokens`** → set to 99th percentile of clean-correct generation lengths
- **Coverage report** in `exclusions.jsonl` → how many items were perturbed successfully per condition
- **Extraction tier distribution** → what fraction of answers were parsed at each tier

### Cell 1 — Clone repo and install CPU dependencies

In [ ]:
import subprocess, sys

subprocess.run(
    ["git", "clone", "https://github.com/natSegOS/glamor-research-onboarding.git"],
    check=True)

%cd glamor-research-onboarding/001_typo_robustness

subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"], check=True)
print("CPU deps installed")

### Cell 2 — Install GPU stack and verify CUDA

`spacy-transformers` is installed here because it uses PyTorch and runs GPU-accelerated
on the T4 during the annotation step (Cell 3).  The `en_core_web_trf` model is the
RoBERTa-based spaCy pipeline used for both pilot and confirmatory annotation runs —
the pilot must be methodologically identical to the main study (design/11 §11.2).

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-gpu.txt", "-q"], check=True)

# Download the spaCy transformer model (GPU-accelerated via spacy-transformers).
# en_core_web_trf uses RoBERTa-base; published benchmarks at spacy.io/models/en.
# The exact model SHA is recorded in data/items/annotation_PROVENANCE.json.
subprocess.run(
    [sys.executable, "-m", "spacy", "download", "en_core_web_trf", "-q"],
    check=True)

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU found — check Runtime > Change runtime type > T4 GPU")

### Cell 3 — Fetch task items and annotate with K_P(x) key terms

**`build_task_items`** downloads 100 items per dataset (GSM-Symbolic p1, GSM8K,
MMLU-Pro, MMLU) from HuggingFace and writes pinned JSONL files to `data/items/`.
The `p1` variant of GSM-Symbolic is used because it carries `question_annotated`
template fields needed for Regime C operand-swap (design/07 §7.x).

**`build_annotated_dataset`** annotates each item with its frozen key-term set
K_P(x) using the `en_core_web_trf` spaCy pipeline (design/04 §4.6).  The output
is written back to the same JSONL files and frozen for the run.

In [ ]:
import subprocess, sys

# Fetch 100 items per dataset from HuggingFace.
# --gsm-config p1: uses the +1-clause GSM-Symbolic variant that carries
# question_annotated fields for Regime C operand-swap.
subprocess.run([
    sys.executable, "tools/build_task_items.py",
    "--reasoning-items", "100",
    "--mcq-items",       "100",
    "--gsm-config",      "p1",
    "--seed",            "1729",
    "--output-directory", "data/items",
], check=True)

# Annotate with frozen K_P(x) key terms using en_core_web_trf.
# GPU-accelerated via spacy-transformers; takes ~5 min on a T4.
subprocess.run([
    sys.executable, "tools/build_annotated_dataset.py",
    "--model-name", "en_core_web_trf",
    "--items-dir",  "data/items",
    "--force",
], check=True)

print("Items annotated and ready in data/items/")

### Cell 4 — Build the SCOWL English dictionary

Downloads SCOWL 2020.12.07 (Kevin Atkinson, wordlist.aspell.net) and builds
`data/wordlists/en_us_pinned.txt`.  SCOWL size band 60 covers standard
dictionary vocabulary without rare, archaic, or technical terms outside
native-speaker competence (Atkinson, SCOWL documentation).

This dictionary is the `is_word` predicate that separates Regime A nonword
typos from Regime B real-word shifts.  The SHA-256 of the source files is
recorded in `data/wordlists/PROVENANCE.json` for audit and pre-registration.

In [ ]:
import subprocess, sys

# Download SCOWL 2020.12.07 from the official GitHub mirror.
subprocess.run([
    "wget", "-q", "-O", "/tmp/scowl.tar.gz",
    "https://github.com/en-wl/wordlist/archive/refs/tags/v2020.12.07.tar.gz",
], check=True)
subprocess.run(["tar", "-xzf", "/tmp/scowl.tar.gz", "-C", "/tmp/"], check=True)

# Build the pinned vocabulary from the size-60 word lists.
# The final/ subdirectory contains pre-built lists named english-words.NN.
subprocess.run([
    sys.executable, "tools/build_dictionary.py",
    "--scowl-path",     "/tmp/wordlist-2020.12.07/final/",
    "--scowl-max-size", "60",
], check=True)

print("Dictionary ready — data/wordlists/en_us_pinned.txt")

### Cell 5 — Generate pilot outputs

Runs all conditions in `configs/pilot.yaml` (Regimes A, B, C) against
`qwen_1b5_pilot` (Qwen2.5-1.5B-Instruct).  Scoring is performed inline:
each generation is immediately scored with the four-way parse-status
classifier (VALID / UNPARSEABLE / CLARIFICATION / REFUSAL) before the
next batch begins.  No separate scoring step is needed.

The runner is idempotent: if interrupted, re-running resumes from the last
completed shard.  Output: `results/pilot/pilot_generations.jsonl`.

Expected time: ~30–60 min on a T4.

In [ ]:
import subprocess, sys

try:
    git_commit = subprocess.run(
        ["git", "rev-parse", "HEAD"], capture_output=True, text=True
    ).stdout.strip() or "unpinned"

    subprocess.run([
        sys.executable, "tools/run_generation.py",
        "--config", "configs/pilot.yaml",
        "--model", "qwen_1b5_pilot",
        "--output-directory", "results/pilot",
        "--dictionary", "data/wordlists/en_us_pinned.txt",
        "--git-commit", git_commit,
    ], check=True, capture_output=True, text=True)
    
    print("generation done — results/pilot/pilot_generations.jsonl")

except subprocess.CalledProcessError as e:
    print(f"\n[ERROR] Script crashed!")
    print(f"Exit Code: {e.returncode}")
    print(f"--- Standard Error (stderr) --- \n{e.stderr}")
    print(f"--- Standard Output (stdout) --- \n{e.stdout}")

### Cell 6 — Analyze results and download

**`run_analysis`** produces the per-cell accuracy table, discordant-pair rates,
mediation estimates, and figures in `analysis/pilot/`.

**`build_report`** writes a self-contained HTML drill-down at
`results/pilot/report.html` with global statistics and a per-item diff view.

Everything is zipped and downloaded as `pilot_results.zip`.

In [1]:
import subprocess, sys, zipfile, pathlib

generations_path = "results/pilot/pilot_generations.jsonl"

# Statistical summary: cell table, discordant rates, mediation, figures.
subprocess.run([
    sys.executable, "tools/run_analysis.py",
    "--generations",      generations_path,
    "--output-directory", "analysis/pilot",
], check=True)

# Self-contained HTML drill-down report.
subprocess.run([
    sys.executable, "tools/build_report.py",
    "--generations", generations_path,
    "--output",      "results/pilot/report.html",
], check=True)

# Zip all outputs and download.
zip_path = pathlib.Path("pilot_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in pathlib.Path("analysis/pilot").rglob("*"):
        if path.is_file():
            zf.write(path)
    for extra in ["results/pilot/report.html",
                  "results/pilot/pilot_generations.jsonl",
                  "results/pilot/pilot_exclusions.jsonl"]:
        if pathlib.Path(extra).exists():
            zf.write(extra)

try:
    from google.colab import files
    files.download(str(zip_path))
    print("Downloaded pilot_results.zip")
except ImportError:
    print(f"Results at {zip_path.resolve()}")

Traceback (most recent call last):
  File "/Users/nathansegura/Development/Personal/glamor-research-onboarding/experiments/001_typo_robustness/tools/run_analysis.py", line 191, in <module>
    main()
    ~~~~^^
  File "/Users/nathansegura/Development/Personal/glamor-research-onboarding/experiments/001_typo_robustness/tools/run_analysis.py", line 165, in main
    rows = load_generation_rows(arguments.generations)
  File "/Users/nathansegura/Development/Personal/glamor-research-onboarding/experiments/001_typo_robustness/src/pipeline/runner.py", line 335, in load_generation_rows
    for line in Path(path).read_text().splitlines():
                ~~~~~~~~~~~~~~~~~~~~^^
  File "/opt/homebrew/Cellar/python@3.14/3.14.6/Frameworks/Python.framework/Versions/3.14/lib/python3.14/pathlib/__init__.py", line 787, in read_text
    with self.open(mode='r', encoding=encoding, errors=errors, newline=newline) as f:
         ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/

CalledProcessError: Command '['/opt/homebrew/Cellar/jupyterlab/4.6.0/libexec/bin/python', 'tools/run_analysis.py', '--generations', 'results/pilot/pilot_generations.jsonl', '--output-directory', 'analysis/pilot']' returned non-zero exit status 1.